# Product Category Prediction

Ova Jupyter radna sveska prikazuje kompletan proces razvoja modela za automatsku predikciju kategorije proizvoda na osnovu naziva proizvoda.

Cilj projekta je da se na osnovu kolone **Product Title** predvidi ciljna kategorija iz kolone **Category Label**.

## 1. Učitavanje biblioteka

Koristimo `pandas` za rad sa podacima, `scikit-learn` za mašinsko učenje i `matplotlib/seaborn` za vizualizaciju rezultata.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

import pickle

## 2. Učitavanje skupa podataka

Učitavamo fajl `products.csv`. Pošto CSV može imati razmake u nazivima kolona, odmah čistimo nazive kolona pomoću `str.strip()`.

In [ ]:
df = pd.read_csv("products.csv")

# Uklanjamo razmake iz naziva kolona, npr. " Category Label" -> "Category Label"
df.columns = df.columns.str.strip()

print("Broj redova i kolona:", df.shape)
print("\nNazivi kolona:")
print(df.columns.tolist())

df.head()

## 3. Osnovna analiza podataka

Proveravamo nedostajuće vrednosti i raspodelu kategorija. Ovo je važno jer kvalitet podataka direktno utiče na kvalitet modela.

In [ ]:
print("Nedostajuće vrednosti po kolonama:")
print(df.isna().sum())

print("\nBroj jedinstvenih kategorija:")
print(df["Category Label"].nunique())

print("\nNajčešće kategorije:")
print(df["Category Label"].value_counts().head(10))

In [ ]:
plt.figure(figsize=(10, 5))
df["Category Label"].value_counts().head(10).plot(kind="bar")
plt.title("Top 10 najčešćih kategorija")
plt.xlabel("Kategorija")
plt.ylabel("Broj proizvoda")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Čišćenje podataka

Uklanjamo redove koji nemaju naziv proizvoda ili kategoriju. Takođe pretvaramo tekst u string format da bismo izbegli greške tokom treniranja modela.

In [ ]:
df = df.dropna(subset=["Product Title", "Category Label"]).copy()

df["Product Title"] = df["Product Title"].astype(str)
df["Category Label"] = df["Category Label"].astype(str)

print("Broj redova posle čišćenja:", len(df))

## 5. Inženjering karakteristika

Za osnovni model koristimo tekstualnu karakteristiku `Product Title`. Dodajemo i nekoliko pomoćnih karakteristika samo za analizu: dužina naslova, broj reči i da li naslov sadrži cifre. One pomažu u razumevanju podataka i mogu biti korisne u budućim verzijama modela.

In [ ]:
df["title_length"] = df["Product Title"].str.len()
df["word_count"] = df["Product Title"].str.split().str.len()
df["has_number"] = df["Product Title"].str.contains(r"\d", regex=True)

df[["Product Title", "Category Label", "title_length", "word_count", "has_number"]].head()

In [ ]:
print(df[["title_length", "word_count"]].describe())

## 6. Podela podataka

Podatke delimo na trening i test skup u odnosu 80:20. Test skup koristimo samo za proveru performansi modela na podacima koje model nije video tokom treniranja.

In [ ]:
X = df["Product Title"]
y = df["Category Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Trening skup:", X_train.shape)
print("Test skup:", X_test.shape)

## 7. Treniranje i poređenje više modela

Testiramo više algoritama:

- Logistic Regression
- Multinomial Naive Bayes
- Linear SVC
- Decision Tree

Svi modeli koriste TF-IDF vektorizaciju, jer algoritmi ne mogu direktno da rade sa tekstom već im je potreban numerički oblik podataka.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVC": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english")),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    results[name] = {
        "pipeline": pipeline,
        "accuracy": accuracy,
        "predictions": y_pred
    }

    print("=" * 60)
    print(name)
    print("Accuracy:", round(accuracy, 4))
    print(classification_report(y_test, y_pred, zero_division=0))

## 8. Vizuelno poređenje tačnosti modela

Grafikon omogućava brzo poređenje performansi svih testiranih modela.

In [ ]:
accuracy_df = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": [results[name]["accuracy"] for name in results]
}).sort_values(by="Accuracy", ascending=False)

accuracy_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(accuracy_df["Model"], accuracy_df["Accuracy"])
plt.title("Poređenje modela prema accuracy")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 9. Matrica zabune za najbolji model

Matrica zabune pokazuje gde model greši, odnosno koje kategorije meša sa drugim kategorijama. Kod većeg broja kategorija matrica može biti velika, pa ovde prikazujemo matricu za najbolji model radi evaluacije.

In [ ]:
best_model_name = accuracy_df.iloc[0]["Model"]
best_pipeline = results[best_model_name]["pipeline"]
best_predictions = results[best_model_name]["predictions"]

print("Najbolji model:", best_model_name)
print("Accuracy:", results[best_model_name]["accuracy"])

cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 10. Čuvanje finalnog modela

Najbolji model čuvamo u fajl `model.pkl`. Taj fajl kasnije koristi skripta `predict_category.py` za interaktivno testiranje novih proizvoda.

In [ ]:
with open("model.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print("Finalni model je sačuvan kao model.pkl")
print("Izabrani model:", best_model_name)

## 11. Ručno testiranje na primerima iz zadatka

Ovim proveravamo kako finalni model reaguje na konkretne nazive proizvoda.

In [ ]:
test_products = [
    "iphone 7 32gb gold,4,3,Apple iPhone 7 32GB",
    "olympus e m10 mark iii geh use silber",
    "kenwood k20mss15 solo",
    "bosch wap28390gb 8kg 1400 spin",
    "bosch serie 4 kgv39vl31g",
    "smeg sbs8004po"
]

for product in test_products:
    prediction = best_pipeline.predict([product])[0]
    print(f"{product} -> {prediction}")

## 12. Zaključak

U ovom projektu razvijen je model za automatsku predikciju kategorije proizvoda na osnovu naziva proizvoda.

Testirano je više algoritama: Logistic Regression, Naive Bayes, Linear SVC i Decision Tree. Modeli su upoređeni na osnovu tačnosti i klasifikacionog izveštaja.

Za finalno rešenje izabran je model sa najboljom tačnošću na test skupu. U praktičnoj primeni najvažnije je da model stabilno prepoznaje kategorije i da se može lako koristiti kroz skriptu `predict_category.py`.

U sledećoj verziji projekta moglo bi se dodatno unaprediti rešenje korišćenjem dodatnih karakteristika kao što su brend, broj reči, prisustvo brojeva i druge informacije iz skupa podataka.